# Solving the Steady-State 2D Radiative Transfer Equation (RTE) using Vanilla-PINNs

In this notebook, we solve the steady-state 2D Radiative Transfer Equation (RTE) in a participating square medium using a **VANILLA-PINN** approach. This case corresponds to the benchmark of **Crosbie and Schrenker [30]** (1984).

### Case 2 (2D)
We focus on a 2D space + 3D direction test case with the following configuration:
* **Geometry:** $0 \le x \le L_x$ and $0 \le y \le L_y$ with $L_x = L_y = 1.0\text{ m}$ (Square medium).
* **Directions:** Direction of propagation $\vec{s}$ defined by spherical angles $(\theta, \phi)$, with directional cosines:
  * $\mu = \sin\theta \cos\phi$ (x-axis)
  * $\eta = \sin\theta \sin\phi$ (y-axis)
* **Equation (2D RTE):**
  $$\mu \frac{\partial I}{\partial x} + \eta \frac{\partial I}{\partial y} + (\kappa + \sigma) I(x, y, \mu, \eta) = \frac{\sigma}{4\pi} G(x, y)$$
  where $G(x, y) = \int_{0}^{2\pi} \int_{0}^{\pi} I(x, y, \theta', \phi') \sin\theta' d\theta' d\phi'$.
* **Physical Properties:**
  * Scattering coefficient: $\sigma = 1.0\text{ m}^{-1}$ (Isotropic scattering).
  * Absorption coefficient: $\kappa = 0\text{ m}^{-1}$ (Non-absorbing scattering medium).
* **Boundary Conditions (BC) on incoming directions:**
  * Top boundary ($y = 1$, for $\eta < 0$): $I(x, 1, \mu, \eta) = 1.0$ (Diffuse radiation).
  * Bottom boundary ($y = 0$, for $\eta > 0$): $I(x, 0, \mu, \eta) = 0.0$
  * Left boundary ($x = 0$, for $\mu > 0$): $I(0, y, \mu, \eta) = 0.0$
  * Right boundary ($x = 1$, for $\mu < 0$): $I(1, y, \mu, \eta) = 0.0$

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. PINN Architecture Definition
Definition of the standard Neural Network (Multi-Layer Perceptron) to approximate the intensity $I(x, y, \mu, \eta)$.

In [2]:
class PinnRFEEq2D(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(4, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 2. Double Quadrature Setup for the Unit Sphere
Setting up a tensor product of Gauss-Legendre quadratures for $\xi = \cos\theta \in [-1, 1]$ and $\phi \in [0, 2\pi]$ to perform solid angle integration.

In [3]:
def init_quadrature(N_theta=8, N_phi=16, device='cpu'):
    nodes_xi, weights_xi = np.polynomial.legendre.leggauss(N_theta)
    nodes_phi, weights_phi = np.polynomial.legendre.leggauss(N_phi)
    nodes_phi = np.pi * (nodes_phi + 1.0)
    weights_phi = np.pi * weights_phi

    quad_mu_list = []
    quad_eta_list = []
    quad_w_list = []

    for i in range(N_theta):
        for j in range(N_phi):
            xi = nodes_xi[i]
            phi = nodes_phi[j]
            w = weights_xi[i] * weights_phi[j]
            
            mu = np.sqrt(1.0 - xi**2) * np.cos(phi)
            eta = np.sqrt(1.0 - xi**2) * np.sin(phi)
            
            quad_mu_list.append(mu)
            quad_eta_list.append(eta)
            quad_w_list.append(w)

    quad_mu = torch.tensor(quad_mu_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_eta = torch.tensor(quad_eta_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_w = torch.tensor(quad_w_list, dtype=torch.float32).view(-1, 1).to(device)
    return quad_mu, quad_eta, quad_w

## 3. Sampling and Collocation Points Generation
Functions to generate collocation points in the 4D domain $(x, y, \mu, \eta)$ and boundary points on the four limits for incoming directions.

In [4]:
def generer_points_collocation(n_pde):
    x = torch.rand(n_pde, 1)
    y = torch.rand(n_pde, 1)
    xi = torch.rand(n_pde, 1) * 2.0 - 1.0
    phi = torch.rand(n_pde, 1) * 2.0 * np.pi
    mu = torch.sqrt(1.0 - xi**2) * torch.cos(phi)
    eta = torch.sqrt(1.0 - xi**2) * torch.sin(phi)
    return x.float(), y.float(), mu.float(), eta.float()

def generer_points_bords(n_bords):
    n_edge = n_bords // 4
    
    # Left edge: x = 0, y in [0,1], mu > 0 (phi in [-pi/2, pi/2])
    x_left = torch.zeros(n_edge, 1)
    y_left = torch.rand(n_edge, 1)
    xi_left = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_left = (torch.rand(n_edge, 1) - 0.5) * np.pi
    mu_left = torch.sqrt(1.0 - xi_left**2) * torch.cos(phi_left)
    eta_left = torch.sqrt(1.0 - xi_left**2) * torch.sin(phi_left)
    
    # Right edge: x = 1, y in [0,1], mu < 0 (phi in [pi/2, 3*pi/2])
    x_right = torch.ones(n_edge, 1)
    y_right = torch.rand(n_edge, 1)
    xi_right = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_right = torch.rand(n_edge, 1) * np.pi + 0.5 * np.pi
    mu_right = torch.sqrt(1.0 - xi_right**2) * torch.cos(phi_right)
    eta_right = torch.sqrt(1.0 - xi_right**2) * torch.sin(phi_right)
    
    # Bottom edge: x in [0,1], y = 0, eta > 0 (phi in [0, pi])
    x_bottom = torch.rand(n_edge, 1)
    y_bottom = torch.zeros(n_edge, 1)
    xi_bottom = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_bottom = torch.rand(n_edge, 1) * np.pi
    mu_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.cos(phi_bottom)
    eta_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.sin(phi_bottom)
    
    # Top edge: x in [0,1], y = 1, eta < 0 (phi in [pi, 2*pi])
    x_top = torch.rand(n_edge, 1)
    y_top = torch.ones(n_edge, 1)
    xi_top = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_top = torch.rand(n_edge, 1) * np.pi + np.pi
    mu_top = torch.sqrt(1.0 - xi_top**2) * torch.cos(phi_top)
    eta_top = torch.sqrt(1.0 - xi_top**2) * torch.sin(phi_top)
    
    return (
        x_left.float(), y_left.float(), mu_left.float(), eta_left.float(),
        x_right.float(), y_right.float(), mu_right.float(), eta_right.float(),
        x_bottom.float(), y_bottom.float(), mu_bottom.float(), eta_bottom.float(),
        x_top.float(), y_top.float(), mu_top.float(), eta_top.float(),
    )

## 4. Loss Functions Definition
Loss calculation for boundary conditions (BC) and the PDE residual (CLP).

In [5]:
def calc_bc_loss(model, 
                 x_l, y_l, mu_l, eta_l,
                 x_r, y_r, mu_r, eta_r,
                 x_b, y_b, mu_b, eta_b,
                 x_t, y_t, mu_t, eta_t):
    
    pred_l = model(torch.cat([x_l, y_l, mu_l, eta_l], dim=1))
    pred_r = model(torch.cat([x_r, y_r, mu_r, eta_r], dim=1))
    pred_b = model(torch.cat([x_b, y_b, mu_b, eta_b], dim=1))
    pred_t = model(torch.cat([x_t, y_t, mu_t, eta_t], dim=1))
    
    loss_l = torch.mean((pred_l - 0.0) ** 2)
    loss_r = torch.mean((pred_r - 0.0) ** 2)
    loss_b = torch.mean((pred_b - 0.0) ** 2)
    loss_t = torch.mean((pred_t - 1.0) ** 2)
    
    return loss_l + loss_r + loss_b + 5.0 * loss_t

def calc_clp_loss(model, x, y, mu, eta, quad_mu, quad_eta, quad_w, kappa=0.0, sigma=1.0):
    x.requires_grad_(True)
    y.requires_grad_(True)
    
    I_pred = model(torch.cat([x, y, mu, eta], dim=1))
    
    I_x = torch.autograd.grad(
        outputs=I_pred,
        inputs=x,
        grad_outputs=torch.ones_like(I_pred),
        create_graph=True,
    )[0]
    
    I_y = torch.autograd.grad(
        outputs=I_pred,
        inputs=y,
        grad_outputs=torch.ones_like(I_pred),
        create_graph=True,
    )[0]
    
    N = x.shape[0]
    N_q = quad_mu.shape[0]
    
    x_expanded = x.repeat(1, N_q)
    y_expanded = y.repeat(1, N_q)
    mu_expanded = quad_mu.t().repeat(N, 1)
    eta_expanded = quad_eta.t().repeat(N, 1)
    
    inputs_quad = torch.stack([x_expanded, y_expanded, mu_expanded, eta_expanded], dim=2).view(-1, 4)
    I_quad_preds = model(inputs_quad).view(N, N_q)
    
    G = torch.sum(I_quad_preds * quad_w.t(), dim=1, keepdim=True)
    
    residual = mu * I_x + eta * I_y + (kappa + sigma) * I_pred - (sigma / (4.0 * np.pi)) * G
    
    loss_pde = torch.mean(residual ** 2)
    return loss_pde

## 5. Hardware (Device), Model, and Optimizer Initialization
Hardware detection, model creation, training data generation and optimizer definition.

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

quad_mu, quad_eta, quad_w = init_quadrature(N_theta=16, N_phi=32, device=device)

n_pde = 15000
n_bords = 4000
kappa = 0.0
sigma = 1.0

x_colloc, y_colloc, mu_colloc, eta_colloc = generer_points_collocation(n_pde)
x_colloc = x_colloc.to(device)
y_colloc = y_colloc.to(device)
mu_colloc = mu_colloc.to(device)
eta_colloc = eta_colloc.to(device)

(
    x_l, y_l, mu_l, eta_l,
    x_r, y_r, mu_r, eta_r,
    x_b, y_b, mu_b, eta_b,
    x_t, y_t, mu_t, eta_t
) = generer_points_bords(n_bords)

x_l, y_l, mu_l, eta_l = x_l.to(device), y_l.to(device), mu_l.to(device), eta_l.to(device)
x_r, y_r, mu_r, eta_r = x_r.to(device), y_r.to(device), mu_r.to(device), eta_r.to(device)
x_b, y_b, mu_b, eta_b = x_b.to(device), y_b.to(device), mu_b.to(device), eta_b.to(device)
x_t, y_t, mu_t, eta_t = x_t.to(device), y_t.to(device), mu_t.to(device), eta_t.to(device)

modele = PinnRFEEq2D(hidden_dim=64, num_layers=4).to(device)

Using device: mps


## 6. Model Training (Adam)
Training phase of the PINN model using the Adam optimizer.

In [7]:
optimizer = optim.Adam(modele.parameters(), lr=0.005)
epochs = 2000

for epoch in range(epochs):
    optimizer.zero_grad()
    loss_bc = calc_bc_loss(modele, 
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc, quad_mu, quad_eta, quad_w, kappa, sigma)
    loss_totale = loss_bc + loss_clp
    loss_totale.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoque {epoch:04d} | "
              f"Loss totale: {loss_totale.item():.2e} | "
              f"CLP: {loss_clp.item():.2e} | "
              f"BC: {loss_bc.item():.2e}")

Epoque 0000 | Loss totale: 1.26e+00 | CLP: 4.28e-03 | BC: 1.24e+00
Epoque 0100 | Loss totale: 1.66e-01 | CLP: 5.77e-03 | BC: 1.37e-01
Epoque 0200 | Loss totale: 1.20e-01 | CLP: 4.10e-03 | BC: 9.94e-02
Epoque 0300 | Loss totale: 6.68e-02 | CLP: 2.15e-03 | BC: 5.60e-02
Epoque 0400 | Loss totale: 5.01e-02 | CLP: 1.77e-03 | BC: 4.12e-02
Epoque 0500 | Loss totale: 3.71e-02 | CLP: 1.57e-03 | BC: 2.93e-02
Epoque 0600 | Loss totale: 2.71e-02 | CLP: 1.26e-03 | BC: 2.08e-02
Epoque 0700 | Loss totale: 2.16e-02 | CLP: 1.04e-03 | BC: 1.64e-02
Epoque 0800 | Loss totale: 1.84e-02 | CLP: 9.11e-04 | BC: 1.38e-02
Epoque 0900 | Loss totale: 1.64e-02 | CLP: 8.50e-04 | BC: 1.22e-02
Epoque 1000 | Loss totale: 1.32e-02 | CLP: 7.00e-04 | BC: 9.69e-03
Epoque 1100 | Loss totale: 1.15e-02 | CLP: 6.38e-04 | BC: 8.33e-03
Epoque 1200 | Loss totale: 1.01e-02 | CLP: 6.12e-04 | BC: 7.08e-03
Epoque 1300 | Loss totale: 9.12e-03 | CLP: 5.58e-04 | BC: 6.33e-03
Epoque 1400 | Loss totale: 8.49e-03 | CLP: 5.55e-04 | BC: 5.71

## 7. Model Training (L-BFGS)
Fine-tuning of the model parameters using L-BFGS.

In [ ]:
def closure():
    optimizer_lbfgs.zero_grad()
    loss_bc = calc_bc_loss(modele, 
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc, quad_mu, quad_eta, quad_w, kappa, sigma)
    loss_totale = loss_bc + loss_clp
    loss_totale.backward()
    return loss_totale

optimizer_lbfgs = optim.LBFGS(modele.parameters(), line_search_fn="strong_wolfe", max_iter=20)
lbfgs_epochs = 500

for epoch in range(lbfgs_epochs):
    loss = optimizer_lbfgs.step(closure)
    if epoch % 20 == 0:
        print(f"Epoque LBFGS {epoch:03d} | Loss totale: {loss.item():.2e}")

Epoque LBFGS 000 | Loss totale: 5.38e-03
Epoque LBFGS 020 | Loss totale: 6.51e-04
Epoque LBFGS 040 | Loss totale: 3.50e-04
Epoque LBFGS 060 | Loss totale: 2.55e-04
Epoque LBFGS 080 | Loss totale: 2.06e-04
Epoque LBFGS 100 | Loss totale: 1.76e-04
Epoque LBFGS 120 | Loss totale: 1.52e-04
Epoque LBFGS 140 | Loss totale: 1.33e-04
Epoque LBFGS 160 | Loss totale: 1.19e-04
Epoque LBFGS 180 | Loss totale: 1.08e-04


KeyboardInterrupt: 

## 8. Visualizing the Results and Comparison with Benchmark

To validate our PINN solution, we compare the predicted incident radiation $G(x, y)$ along the centerlines with the exact benchmark values of **Crosbie and Schrenker (1984)**.

### Origin of the Reference Points
* **Reference Paper:** A. L. Crosbie and R. G. Schrenker, *"Radiative Transfer in a Two-Dimensional Rectangular Medium Exposed to Diffuse Radiation"*, JQSRT, Vol. 31, No. 4, pp. 339-372, 1984.
* **Methodology:** The reference points are obtained from high-accuracy numerical resolution of the singularity-free integral equation of radiative transfer (computed via high-order Gaussian quadrature).


In [9]:
def evaluer_G(model, x_grid, y_grid, quad_mu, quad_eta, quad_w, device):
    Nx = len(x_grid)
    Ny = len(y_grid)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    x_tensor = torch.tensor(X.ravel(), dtype=torch.float32).view(-1, 1).to(device)
    y_tensor = torch.tensor(Y.ravel(), dtype=torch.float32).view(-1, 1).to(device)
    
    N = x_tensor.shape[0]
    N_q = quad_mu.shape[0]
    
    x_expanded = x_tensor.repeat(1, N_q)
    y_expanded = y_tensor.repeat(1, N_q)
    mu_expanded = quad_mu.t().repeat(N, 1)
    eta_expanded = quad_eta.t().repeat(N, 1)
    
    inputs = torch.stack([x_expanded, y_expanded, mu_expanded, eta_expanded], dim=2).view(-1, 4)
    
    G_list = []
    batch_size = 1000
    with torch.no_grad():
        for i in range(0, N, batch_size):
            start_idx = i * N_q
            end_idx = min((i + batch_size) * N_q, N * N_q)
            inputs_batch = inputs[start_idx:end_idx]
            cur_N = inputs_batch.shape[0] // N_q
            I_preds = model(inputs_batch).view(cur_N, N_q)
            G_batch = torch.sum(I_preds * quad_w.t(), dim=1)
            G_list.append(G_batch)
        G_tensor = torch.cat(G_list)
        
    return G_tensor.cpu().numpy().reshape(Y.shape)

x_vals = np.linspace(0.0, 1.0, 100)
y_vals = np.linspace(0.0, 1.0, 100)
G_pred = evaluer_G(modele, x_vals, y_vals, quad_mu, quad_eta, quad_w, device)

# Plot Heatmap of G(x, y)
fig, ax = plt.subplots(figsize=(7, 6))
X, Y = np.meshgrid(x_vals, y_vals)
im = ax.pcolormesh(X, Y, G_pred, cmap='jet', shading='auto')
ax.set_title("Incident Radiation $G(x, y)$ (2D PINN)")
ax.set_xlabel("Position $x$")
ax.set_ylabel("Position $y$")
fig.colorbar(im, label="Incident Radiation $G$")
plt.tight_layout()
fig.savefig("intensity_heatmap_cas2.png", dpi=150)
plt.show()

# Plot Line Profiles
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Reference data from Crosbie & Schrenker (1984)
ref_y = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ref_G_vertical = np.array([2.07, 2.39, 2.76, 3.14, 3.52, 4.06, 4.52, 5.15, 5.91, 6.91, 8.23])

ref_x = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
ref_G_horizontal = np.array([1.72, 3.14, 4.06, 3.14, 1.72])

# Line x = 0.5
y_vals_line = np.linspace(0.0, 1.0, 100)
G_x05 = evaluer_G(modele, np.array([0.5]), y_vals_line, quad_mu, quad_eta, quad_w, device).flatten()
axes[0].plot(y_vals_line, G_x05, color='red', linewidth=2, label="PINN")
axes[0].plot(ref_y, ref_G_vertical, 'ko', label="Crosbie & Schrenker (1984)")
axes[0].set_title("Incident Radiation $G(0.5, y)$")
axes[0].set_xlabel("Position $y$")
axes[0].set_ylabel("$G(0.5, y)$")
axes[0].grid(True)
axes[0].legend()

# Line y = 0.5
x_vals_line = np.linspace(0.0, 1.0, 100)
G_y05 = evaluer_G(modele, x_vals_line, np.array([0.5]), quad_mu, quad_eta, quad_w, device).flatten()
axes[1].plot(x_vals_line, G_y05, color='blue', linewidth=2, label="PINN")
axes[1].plot(ref_x, ref_G_horizontal, 'ko', label="Crosbie & Schrenker (1984)")
axes[1].set_title("Incident Radiation $G(x, 0.5)$")
axes[1].set_xlabel("Position $x$")
axes[1].set_ylabel("$G(x, 0.5)$")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
fig.savefig("intensity_boundaries_cas2.png", dpi=150)
plt.show()

KeyboardInterrupt: 